In [ ]:
import os
import yaml

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
from pathlib import Path

from reader.io_utils import load_processed_datasets
from modules.wrf_rio import wrf_prepare_for_rio

# ==========================================================
# User settings
# ==========================================================

init_date = "2021012700"
lead_time=6

varname = 'qpf'
domain_name = 'Central_CA'
source='archive'


domain = {
    "lon_min": -125.,
    "lon_max": -110.,
    "lat_min": 25.,
    "lat_max": 50.,
}

# ---------------------------------------------------------
# Load processed datasets
# ---------------------------------------------------------

fc, final_ds = (
    load_processed_datasets(
        init_date=init_date,
        source=source,
    )
)

# ---------------------------------------------------------
# Load Burn Scar Shapefile
# ---------------------------------------------------------

fp = globalvars.path_to_data + 'downloads/calfire/California_Fire_Perimeters.shp'
polys = gpd.read_file(fp)

keep_names = ["CZU LIGHTNING COMPLEX"]
polys = polys[polys["FIRE_NAME"].isin(keep_names)]

# ---------------------------------------------------------
# Subset to Burn Scar
# ---------------------------------------------------------
field = final_ds[varname]
# prepare for rioxarray
da = wrf_prepare_for_rio(field)

# reproject shapefile to match WRF CRS
polys = polys.to_crs(da.rio.crs)

print(f"\nVariable: {varname}, Model: {model}")
print(f"  Raster CRS: {da.rio.crs}")
print(f"  Polygon CRS: {polys.crs}")
print(f"  Raster bounds: {da.rio.bounds()}")
print(f"  Polygon bounds: {polys.geometry.bounds}")

results = []
# --- loop through polygons ---
for _, poly in polys.iterrows():
    # Clip both datasets to polygon
    da_clip = da.rio.clip([poly.geometry], polys.crs, drop=True)


    # Compute stats
    stats = lambda da: {
        "mean": float(da.mean().values),
        "min": float(da.min().values),
        "max": float(da.max().values)
    }

    da_stats = stats(da_clip)

    results.append({
        "variable": varname,
        "burnscar": poly["Name"],
        "mean_model": da_stats["mean"],
        "min_model": da_stats["min"],
        "max_model": da_stats["max"],
    })

df = pd.DataFrame(results)

In [2]:
import xarray as xr
fname = "/home/dnash/comet_data/preprocessed/NRT_WWRF/wwrf_qpf_2021012200.nc"
ds = xr.open_dataset(fname)
ds

<xarray.Dataset>
Dimensions:   (time: 169, ensemble: 24, south_north: 569, west_east: 618)
Coordinates:
  * time      (time) datetime64[ns] 2021-01-22 ... 2021-01-29
  * ensemble  (ensemble) object 'ecm00' 'ecm01' 'ecm02' ... 'ecm22' 'ecm23'
    XLAT      (ensemble, time, south_north, west_east) float32 ...
    XLONG     (ensemble, time, south_north, west_east) float32 ...
Dimensions without coordinates: south_north, west_east
Data variables:
    RAIN      (ensemble, time, south_north, west_east) float32 ...
Attributes: (12/85)
    TITLE:                            OUTPUT FROM WRF V4.2 MODEL
    START_DATE:                      2021-01-22_00:00:00
    WEST-EAST_GRID_DIMENSION:        619
    SOUTH-NORTH_GRID_DIMENSION:      570
    BOTTOM-TOP_GRID_DIMENSION:       60
    DX:                              9000.0
    ...                              ...
    ISLAKE:                          21
    ISICE:                           15
    ISURBAN:                         13
    ISOILWATER:                      14
    HYBRID_OPT:                      2
    ETAC:                            0.2

In [3]:
print(ds.attrs)

{'TITLE': ' OUTPUT FROM WRF V4.2 MODEL', 'START_DATE': '2021-01-22_00:00:00', 'WEST-EAST_GRID_DIMENSION': 619, 'SOUTH-NORTH_GRID_DIMENSION': 570, 'BOTTOM-TOP_GRID_DIMENSION': 60, 'DX': 9000.0, 'DY': 9000.0, 'AERCU_OPT': 0, 'AERCU_FCT': 1.0, 'IDEAL_CASE': 0, 'DIFF_6TH_SLOPEOPT': 0, 'AUTO_LEVELS_OPT': 2, 'DIFF_6TH_THRESH': 0.1, 'DZBOT': 50.0, 'DZSTRETCH_S': 1.3, 'DZSTRETCH_U': 1.1, 'GRIDTYPE': 'C', 'DIFF_OPT': 1, 'KM_OPT': 4, 'DAMP_OPT': 3, 'DAMPCOEF': 0.2, 'KHDIF': 0.0, 'KVDIF': 0.0, 'MP_PHYSICS': 10, 'RA_LW_PHYSICS': 4, 'RA_SW_PHYSICS': 4, 'SF_SFCLAY_PHYSICS': 1, 'SF_SURFACE_PHYSICS': 2, 'BL_PBL_PHYSICS': 1, 'CU_PHYSICS': 3, 'SF_LAKE_PHYSICS': 0, 'SURFACE_INPUT_SOURCE': 1, 'SST_UPDATE': 0, 'GRID_FDDA': 0, 'GFDDA_INTERVAL_M': 0, 'GFDDA_END_H': 0, 'GRID_SFDDA': 0, 'SGFDDA_INTERVAL_M': 0, 'SGFDDA_END_H': 0, 'HYPSOMETRIC_OPT': 2, 'USE_THETA_M': 1, 'GWD_OPT': 0, 'SF_URBAN_PHYSICS': 0, 'SF_SURFACE_MOSAIC': 0, 'SF_OCEAN_PHYSICS': 0, 'WEST-EAST_PATCH_START_UNSTAG': 1, 'WEST-EAST_PATCH_END_UNST

In [ ]:
# ---------------------------------------------------------
# Plot Meteogram (x-axis lead time, y-axis ensemble members
# ---------------------------------------------------------